# Train and export the original Lorentz Hyperbolic GNN

This notebook uses the original Lorentz-model detector from this repository and your real Elliptic Bitcoin CSVs in Google Drive. It trains once, then exports transaction scores, embeddings, and test metrics for the Streamlit app.

**Important:** the checkpoint from the earlier deployment iteration is a different Poincaré-ball architecture, so it is intentionally not reused. This run creates a checkpoint compatible with the original Lorentz model. No synthetic transaction data is used. A GPU runtime is recommended.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import zipfile
from google.colab import drive, files

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/elliptic dataset')
REPO = Path('/content/Hyperbolic_GNNs')
TRAIN_DIR = BASE / 'lorentz_training_artifacts'
EXPORT_DIR = BASE / 'lorentz_deployment_export'
required = {'elliptic_txs_features.csv', 'elliptic_txs_classes.csv', 'elliptic_txs_edgelist.csv'}
candidates = [p.parent for p in BASE.rglob('elliptic_txs_features.csv') if required.issubset({x.name for x in p.parent.iterdir() if x.is_file()})]
if len(candidates) != 1:
    raise FileNotFoundError(f'Expected one complete Elliptic dataset below {BASE}; found: {candidates}')
DATA_DIR = candidates[0]
print('Dataset:', DATA_DIR)
print('Training outputs:', TRAIN_DIR)
print('Deployment export:', EXPORT_DIR)

In [ ]:
# Get the deployment source that was pushed to GitHub before starting this notebook.
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', 'https://github.com/BSaiCharan-GH/Hyperbolic_GNNs.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
if not (REPO / 'train.py').is_file() or not (REPO / 'export_predictions.py').is_file():
    raise FileNotFoundError('Deployment source is not synced to GitHub main yet. Run publish_to_github.ps1 from the downloaded bundle first.')
missing = [name for name in ('torch_geometric', 'geoopt', 'sklearn') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', 'geoopt', 'scikit-learn', 'pandas', 'numpy'], check=True)
print('Runtime ready. CUDA available:', __import__('torch').cuda.is_available())

## Train the original model

This is the required training step because the earlier checkpoint uses a different model architecture. The training code uses labelled timesteps 1–34 for training, 35–41 for validation and threshold selection, and 42–49 for held-out testing. All graph edges are used during message passing, so the evaluation is transductive rather than future-isolated.

In [ ]:
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(REPO / 'train.py'), '--data-dir', str(DATA_DIR), '--epochs', '30', '--patience', '6', '--device', 'auto', '--output-dir', str(TRAIN_DIR)]
print('Starting a new run of the original Lorentz GNN. This may take time on the full graph.')
subprocess.run(cmd, cwd=REPO, check=True)
print('Checkpoint:', TRAIN_DIR / 'forensic_detector.pt')
print('Metrics:', TRAIN_DIR / 'metrics.json')
print('History:', TRAIN_DIR / 'training_history.json')

## Export model outputs for the web app

The exporter loads the checkpoint created in the previous cell. It does not retrain. The full transaction output includes the original transaction ID, timestep, known/unknown label, illicit score, review band, graph degree and two-dimensional embedding.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(REPO / 'export_predictions.py'), '--data-dir', str(DATA_DIR), '--checkpoint', str(TRAIN_DIR / 'forensic_detector.pt'), '--output-dir', str(EXPORT_DIR), '--device', 'auto']
subprocess.run(cmd, cwd=REPO, check=True)
# Include the training curve in the downloaded artifact bundle.
import shutil
shutil.copy2(TRAIN_DIR / 'training_history.json', EXPORT_DIR / 'training_history.json')
bundle = BASE / 'lorentz_deployment_artifacts.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in ('predictions.csv', 'deployment_metrics.json', 'training_history.json'):
        zf.write(EXPORT_DIR / name, arcname=name)
print('Created:', bundle)
print('Now downloading the three real model-output artifacts.')
files.download(str(bundle))

## Finalise the public app

1. Extract `lorentz_deployment_artifacts.zip` into the deployment bundle's `artifacts/` folder. The folder should contain `predictions.csv`, `deployment_metrics.json`, and `training_history.json`.
2. Run `publish_to_github.ps1` again from the same folder so these genuine outputs are committed to the repository. Do not upload any raw dataset files or the full training checkpoint.
3. In Streamlit Community Cloud, create the app using `BSaiCharan-GH/Hyperbolic_GNNs`, branch `main`, main file `app.py`.

Keep the training checkpoint and raw dataset in Drive. Report the metrics the notebook actually prints; do not substitute results from the previous Poincaré model.